# Cyclistic PySpark Prototype

Prototype analysis that mirrors `danny.ipynb` using PySpark instead of pandas.
- DataFrame API only — no RDD, no `.rdd.map()`
- `df.show()` output mirrors the pandas prints in `danny.ipynb`
- Data: `data/processed/trips_clean.csv`, limited to 10 000 rows (same as `nrows=10000` in `danny.ipynb`)

> **Note:** `data/processed/trips_clean.csv` does not include `rideable_type` or lat/lng columns,
> so the *Bike type usage* section from `danny.ipynb` is omitted here.

In [ ]:
%pip install pyspark==3.5.1 python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


## Start Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp,
    hour, date_format,
    count, avg, stddev, min, max,
    percentile_approx, when, round as spark_round,
)

spark = (
    SparkSession.builder
    .appName("CyclisticSparkDanny")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 3.5.1


## Load data

Read the first 10 000 rows — same as `pd.read_csv(..., nrows=10000)` in danny.

In [ ]:
DATA_PATH = "data/processed/trips_clean.csv"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(DATA_PATH)
    .limit(10000)
    .cache()  # cache early — we'll reuse this slice many times
)

print("Columns:", df.columns)
df.show(5, truncate=False)

Columns: ['ride_id', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'member_casual']
+----------------+-------------------+-------------------+----------------------------+----------------+---------------------------+--------------+-------------+
|ride_id         |started_at         |ended_at           |start_station_name          |start_station_id|end_station_name           |end_station_id|member_casual|
+----------------+-------------------+-------------------+----------------------------+----------------+---------------------------+--------------+-------------+
|A847FADBBC638E45|2020-04-26 17:45:14|2020-04-26 18:12:03|Eckhart Park                |86              |Lincoln Ave & Diversey Pkwy|152.0         |member       |
|5405B80E996FF60D|2020-04-17 17:08:54|2020-04-17 17:17:03|Drake Ave & Fullerton Ave   |503             |Kosciuszko Park            |499.0         |member       |
|5DD24A79A4E006F4|2020-04-01 17:54:13|2020-04-

## Calculate ride duration

In [ ]:
df = (
    df
    .withColumn("started_at", to_timestamp("started_at", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ended_at",   to_timestamp("ended_at",   "yyyy-MM-dd HH:mm:ss"))
    .withColumn(
        "ride_length_minutes",
        (unix_timestamp("ended_at") - unix_timestamp("started_at")) / 60.0,
    )
)

df.select("ride_id", "started_at", "ended_at", "ride_length_minutes").show(5, truncate=False)

+----------------+-------------------+-------------------+-------------------+
|ride_id         |started_at         |ended_at           |ride_length_minutes|
+----------------+-------------------+-------------------+-------------------+
|A847FADBBC638E45|2020-04-26 17:45:14|2020-04-26 18:12:03|26.816666666666666 |
|5405B80E996FF60D|2020-04-17 17:08:54|2020-04-17 17:17:03|8.15               |
|5DD24A79A4E006F4|2020-04-01 17:54:13|2020-04-01 18:08:36|14.383333333333333 |
|2A59BBDF5CDBA725|2020-04-07 12:50:19|2020-04-07 13:02:31|12.2               |
|27AD306C119C6158|2020-04-18 10:22:59|2020-04-18 11:15:54|52.916666666666664 |
+----------------+-------------------+-------------------+-------------------+
only showing top 5 rows



## Quality checks

Looking for missing values.  
Filter out any negative or >24 hr rides.

In [ ]:
# --- Missing values per column ---
print("Missing values per column:")
df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
).show()

# --- Ride length summary (matches pandas .describe()) ---
print("\nRide length (minutes) summary:")
df.select("ride_length_minutes").describe().show()

# --- Filter ---
MAX_DURATION = 24 * 60  # 1 440 minutes

print("Number of rides before filtering:", df.count())

df_clean = (
    df.filter(
        (col("ride_length_minutes") > 0)
        & (col("ride_length_minutes") <= MAX_DURATION)
    )
    .cache()
)

print("Number of rides after filtering:",  df_clean.count())

Missing values per column:
+-------+----------+--------+------------------+----------------+----------------+--------------+-------------+-------------------+
|ride_id|started_at|ended_at|start_station_name|start_station_id|end_station_name|end_station_id|member_casual|ride_length_minutes|
+-------+----------+--------+------------------+----------------+----------------+--------------+-------------+-------------------+
|      0|         0|       0|                 0|               0|              13|            13|            0|                  0|
+-------+----------+--------+------------------+----------------+----------------+--------------+-------------+-------------------+


Ride length (minutes) summary:
+-------+--------------------+
|summary| ride_length_minutes|
+-------+--------------------+
|  count|               10000|
|   mean|  37.846839999999965|
| stddev|    518.614897594109|
|    min|-0.23333333333333334|
|    max|  36918.833333333336|
+-------+--------------------+



## Member vs casual

Compare ride lengths and trip counts between members and casual riders.

In [ ]:
# --- Ride length stats by user type (mirrors pandas .describe()) ---
print("Ride length (minutes) by user type:")
df_clean.groupBy("member_casual").agg(
    count("ride_length_minutes").alias("count"),
    spark_round(avg("ride_length_minutes"),   6).alias("mean"),
    spark_round(stddev("ride_length_minutes"), 6).alias("std"),
    min("ride_length_minutes").alias("min"),
    percentile_approx("ride_length_minutes", 0.25).alias("25%"),
    percentile_approx("ride_length_minutes", 0.50).alias("50%"),
    percentile_approx("ride_length_minutes", 0.75).alias("75%"),
    max("ride_length_minutes").alias("max"),
).show()

# --- Trip share by user type ---
total = df_clean.count()
print("\nTrip share by user type:")
(
    df_clean.groupBy("member_casual")
    .count()
    .withColumn("share_%", spark_round(col("count") / total * 100, 2))
    .orderBy(col("count").desc())
    .show()
)

Ride length (minutes) by user type:
+-------------+-----+---------+---------+-------------------+------------------+------------------+------------------+------------------+
|member_casual|count|     mean|      std|                min|               25%|               50%|               75%|               max|
+-------------+-----+---------+---------+-------------------+------------------+------------------+------------------+------------------+
|       member| 7038|18.052214|37.970891|0.03333333333333333|               7.0|13.433333333333334|23.666666666666668|1406.8666666666666|
|       casual| 2940|41.995255|87.002078|0.08333333333333333|14.183333333333334|25.066666666666666|             43.25|           1355.95|
+-------------+-----+---------+---------+-------------------+------------------+------------------+------------------+------------------+


Trip share by user type:
+-------------+-----+-------+
|member_casual|count|share_%|
+-------------+-----+-------+
|       member| 703

## Time and day patterns

Examine how rides vary by hour of day and day of week for members vs casual riders.

In [ ]:
df_clean = (
    df_clean
    .withColumn("hour",        hour("started_at"))
    .withColumn("day_of_week", date_format("started_at", "EEEE"))
)

df_clean.select("started_at", "hour", "day_of_week").show(5, truncate=False)

+-------------------+----+-----------+
|started_at         |hour|day_of_week|
+-------------------+----+-----------+
|2020-04-26 17:45:14|17  |Sunday     |
|2020-04-17 17:08:54|17  |Friday     |
|2020-04-01 17:54:13|17  |Wednesday  |
|2020-04-07 12:50:19|12  |Tuesday    |
|2020-04-18 10:22:59|10  |Saturday   |
+-------------------+----+-----------+
only showing top 5 rows



In [ ]:
# Trips by hour of day and user type
print("Trips by hour of day and user type:")
(
    df_clean
    .groupBy("hour", "member_casual")
    .count()
    .orderBy("hour", "member_casual")
    .show(48)
)

Trips by hour of day and user type:
+----+-------------+-----+
|hour|member_casual|count|
+----+-------------+-----+
|   0|       casual|    9|
|   0|       member|   19|
|   1|       casual|    7|
|   1|       member|   12|
|   2|       casual|    6|
|   2|       member|   14|
|   3|       casual|    1|
|   3|       member|    5|
|   4|       casual|    2|
|   4|       member|   17|
|   5|       casual|    5|
|   5|       member|   60|
|   6|       casual|   21|
|   6|       member|  164|
|   7|       casual|   43|
|   7|       member|  278|
|   8|       casual|   49|
|   8|       member|  283|
|   9|       casual|   63|
|   9|       member|  241|
|  10|       casual|  122|
|  10|       member|  238|
|  11|       casual|  205|
|  11|       member|  426|
|  12|       casual|  273|
|  12|       member|  539|
|  13|       casual|  315|
|  13|       member|  565|
|  14|       casual|  347|
|  14|       member|  622|
|  15|       casual|  351|
|  15|       member|  640|
|  16|       casual

In [ ]:
# Trips by day of week and user type
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Use a manual sort index so days appear Mon-Sun, not alphabetically
day_index = F.create_map(*[v for day in DAY_ORDER for v in (F.lit(day), F.lit(DAY_ORDER.index(day)))])

print("Trips by day of week and user type:")
(
    df_clean
    .groupBy("day_of_week", "member_casual")
    .count()
    .withColumn("_sort", day_index[col("day_of_week")])
    .orderBy("_sort", "member_casual")
    .drop("_sort")
    .show(14)
)

Trips by day of week and user type:
+-----------+-------------+-----+
|day_of_week|member_casual|count|
+-----------+-------------+-----+
|     Monday|       casual|  316|
|     Monday|       member|  961|
|    Tuesday|       casual|  477|
|    Tuesday|       member| 1062|
|  Wednesday|       casual|  211|
|  Wednesday|       member|  777|
|   Thursday|       casual|  301|
|   Thursday|       member| 1070|
|     Friday|       casual|  329|
|     Friday|       member|  888|
|   Saturday|       casual|  494|
|   Saturday|       member|  999|
|     Sunday|       casual|  812|
|     Sunday|       member| 1281|
+-----------+-------------+-----+



## Station-based insights

In [ ]:
for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    (
        df_clean
        .filter(col("member_casual") == user_type)
        .filter(col("start_station_name").isNotNull())
        .groupBy("start_station_name")
        .count()
        .orderBy(col("count").desc())
        .show(10, truncate=False)
    )

    print(f"\nTop 10 end stations for {user_type}s:")
    (
        df_clean
        .filter(col("member_casual") == user_type)
        .filter(col("end_station_name").isNotNull())
        .groupBy("end_station_name")
        .count()
        .orderBy(col("count").desc())
        .show(10, truncate=False)
    )


Top 10 start stations for members:
+-------------------------------+-----+
|start_station_name             |count|
+-------------------------------+-----+
|Clark St & Elm St              |76   |
|Pine Grove Ave & Irving Park Rd|69   |
|Dearborn St & Erie St          |68   |
|McClurg Ct & Erie St           |65   |
|Sheridan Rd & Irving Park Rd   |65   |
|Clark St & Chicago Ave         |63   |
|Desplaines St & Kinzie St      |57   |
|Ashland Ave & Division St      |57   |
|Wells St & Concord Ln          |56   |
|Broadway & Barry Ave           |53   |
+-------------------------------+-----+
only showing top 10 rows


Top 10 end stations for members:
+-------------------------------+-----+
|end_station_name               |count|
+-------------------------------+-----+
|Dearborn St & Erie St          |72   |
|Clark St & Elm St              |68   |
|Desplaines St & Kinzie St      |61   |
|Sheridan Rd & Irving Park Rd   |61   |
|Indiana Ave & Roosevelt Rd     |60   |
|Clark St & Armitage Ave

In [ ]:
# Average ride length by start station and user type (stations with >= 30 trips)
print("Average ride length (minutes) by start station and user type (>= 30 trips):")
(
    df_clean
    .filter(col("start_station_name").isNotNull())
    .groupBy("start_station_name", "member_casual")
    .agg(
        count("ride_id").alias("count"),
        spark_round(avg("ride_length_minutes"), 2).alias("mean"),
    )
    .filter(col("count") >= 30)
    .orderBy(col("mean").desc())
    .show(20, truncate=False)
)

Average ride length (minutes) by start station and user type (>= 30 trips):
+------------------------------+-------------+-----+-----+
|start_station_name            |member_casual|count|mean |
+------------------------------+-------------+-----+-----+
|Clark St & Wellington Ave     |member       |37   |48.19|
|Clark St & Schiller St        |member       |46   |42.68|
|Clark St & Elm St             |casual       |32   |28.18|
|Millennium Park               |member       |36   |27.9 |
|Southport Ave & Irving Park Rd|member       |32   |24.56|
|Southport Ave & Clark St      |member       |33   |23.49|
|Clark St & Randolph St        |member       |45   |23.37|
|Walsh Park                    |member       |30   |22.31|
|Halsted St & Roscoe St        |member       |32   |21.17|
|Sheffield Ave & Wellington Ave|member       |49   |21.12|
|Wells St & Evergreen Ave      |member       |53   |20.74|
|Sheffield Ave & Waveland Ave  |member       |38   |20.5 |
|Lake Shore Dr & North Blvd    |member 

## Segment comparisons for business insights

Weekend vs weekday, commute vs leisure, and ride length bands.

In [ ]:
COMMUTE_HOURS = list(range(6, 10)) + list(range(16, 20))

df_clean = (
    df_clean
    .withColumn("is_weekend",      col("day_of_week").isin(["Saturday", "Sunday"]))
    .withColumn("is_commute_hour", col("hour").isin(COMMUTE_HOURS))
    .withColumn(
        "duration_band",
        when(col("ride_length_minutes") <= 15, "short (<=15m)")
        .when(col("ride_length_minutes") <= 45, "medium (15-45m)")
        .otherwise("long (>45m)"),
    )
)

# Weekend vs weekday
print("Weekend vs weekday trips by user type:")
(
    df_clean
    .groupBy("member_casual", "is_weekend")
    .count()
    .orderBy("member_casual", "is_weekend")
    .show()
)

# Commute vs non-commute
print("\nCommute-hour vs other-hour trips by user type:")
(
    df_clean
    .groupBy("member_casual", "is_commute_hour")
    .count()
    .orderBy("member_casual", "is_commute_hour")
    .show()
)

# Ride length bands as % of trips per user type
band_counts = df_clean.groupBy("member_casual", "duration_band").count()
totals      = (
    df_clean.groupBy("member_casual")
    .count()
    .withColumnRenamed("count", "total")
)

print("\nRide length bands (% of trips) by user type:")
(
    band_counts
    .join(totals, "member_casual")
    .withColumn("pct_%", spark_round(col("count") / col("total") * 100, 1))
    .select("member_casual", "duration_band", "pct_%")
    .orderBy("member_casual", "duration_band")
    .show()
)

Weekend vs weekday trips by user type:
+-------------+----------+-----+
|member_casual|is_weekend|count|
+-------------+----------+-----+
|       casual|     false| 1634|
|       casual|      true| 1306|
|       member|     false| 4758|
|       member|      true| 2280|
+-------------+----------+-----+


Commute-hour vs other-hour trips by user type:
+-------------+---------------+-----+
|member_casual|is_commute_hour|count|
+-------------+---------------+-----+
|       casual|          false| 1813|
|       casual|           true| 1127|
|       member|          false| 3572|
|       member|           true| 3466|
+-------------+---------------+-----+


Ride length bands (% of trips) by user type:
+-------------+---------------+-----+
|member_casual|  duration_band|pct_%|
+-------------+---------------+-----+
|       casual|    long (>45m)| 24.1|
|       casual|medium (15-45m)| 48.7|
|       casual|  short (<=15m)| 27.2|
|       member|    long (>45m)|  2.6|
|       member|medium (15-45m)|